# Voice Assistant Testing Framework - Part 1: Preparation and Setup
## Jetson Orin Nano 8GB - LLM with RAG vs LLM Standalone Performance Comparison

**Researcher:** aRJey  
**Date:** 2025  
**Platform:** NVIDIA Jetson Orin Nano 8GB  
**JetPack:** 6.2.1 | CUDA: 12.6.77 | TensorRT: 10.7.0.23 | cuDNN: 9.17.1.4

---

### Overview
Notebook ini adalah bagian pertama dari serangkaian notebook untuk menguji performa sistem voice assistant dengan dua mode:
1. **LLM + RAG (Retrieval-Augmented Generation)** - LLM dengan akses ke knowledge base
2. **LLM Standalone** - LLM tanpa RAG

### Tujuan Notebook Ini:
- Setup environment dan dependencies
- Load dan prepare dokumen untuk RAG
- Prepare test questions
- Verify system components
- Create baseline measurements

---
## 1. Import Libraries dan Setup Environment

In [ ]:
import sys
import os
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Tuple
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ Libraries imported successfully")
print(f"Python version: {sys.version}")
print(f"Working directory: {os.getcwd()}")

---
## 2. Setup Paths dan Configuration

In [ ]:
# Define paths
PROJECT_DIR = Path('/home/claude')  # Adjust to your actual path on Jetson
DATA_DIR = PROJECT_DIR / 'data'
RESULTS_DIR = PROJECT_DIR / 'results'
LOGS_DIR = PROJECT_DIR / 'logs'
AUDIO_DIR = PROJECT_DIR / 'test_audio'

# Create directories if they don't exist
for dir_path in [DATA_DIR, RESULTS_DIR, LOGS_DIR, AUDIO_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)
    print(f"✓ Directory ready: {dir_path}")

# Configuration
CONFIG = {
    'platform': 'Jetson Orin Nano 8GB',
    'jetpack_version': '6.2.1',
    'cuda_version': '12.6.77',
    'tensorrt_version': '10.7.0.23',
    'cudnn_version': '9.17.1.4',
    'whisper_model': 'tiny',
    'llm_model': 'llama3.2:3b',
    'embedding_model': 'all-MiniLM-L6-v2',
    'test_date': datetime.now().strftime('%Y-%m-%d'),
    'num_test_repetitions': 3  # Each question tested 3 times for statistical robustness
}

print("\n📋 Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

---
## 3. Load Test Questions

In [ ]:
# Define test questions based on your List_Pertanyaan.docx
# Using English version for consistency

test_questions = {
    'simple': [
        "What is the main function of a resistor in an electronic circuit?",
        "Name three examples of sensors commonly used with Arduino microcontrollers.",
        "What does LED stand for, and how do you safely connect it to a voltage source?",
        "What is the basic difference between Voltage and Current?",
        "Which microcontroller is more commonly used by beginners, Arduino Uno or ESP32? Give one reason."
    ],
    'complex': [
        "Conceptually explain how Pulse Width Modulation (PWM) technique can be used to adjust LED brightness or DC motor speed. What is the relationship between duty cycle and the resulting output?",
        "In the context of serial communication between a microcontroller and a computer, what are the fundamental differences between the UART and I2C protocols in terms of connection topology, number of wires, and speed?",
        "A circuit uses a linear regulator LM7805 to step down voltage from 12V to 5V. If the load draws 500mA of current, calculate the approximate power dissipated as heat in the regulator. Why is a switching regulator (like LM2596) more efficient for this case?",
        "You want to read an analog voltage from a potentiometer using an Arduino. Describe the process flow from the voltage at the analog pin until it becomes a digital value (ADC), including the concept of resolution (e.g., 10-bit).",
        "What is meant by Interrupt in microcontroller programming? Provide a practical example where using an interrupt is more effective than polling when reading input from a button."
    ],
    'true_false_basic': [
        "Capacitors can store electric charge and are often used to smooth out voltage ripples. True or False?",
        "An NPN transistor turns ON when the base voltage is lower than the emitter voltage. True or False?",
        "Digital pins on a microcontroller can only output HIGH (5V) or LOW (0V). True or False?",
        "Ohm's Law states that Current (I) is directly proportional to Voltage (V) and inversely proportional to Resistance (R), formulated as I = V x R. True or False?",
        "The higher the resistor value connected in series with an LED, the dimmer the LED will glow. True or False?"
    ],
    'true_false_advanced': [
        "The ESP32 platform from Espressif not only has WiFi and Bluetooth capabilities but also supports programming with MicroPython and Rust, in addition to the Arduino IDE. True or False?",
        "In modern IoT development, the MQTT protocol has completely replaced HTTP for device-to-cloud communication due to its much higher speed. True or False?",
        "To connect external sensors or modules to a microcontroller, breakout boards are often used because they already include the necessary support components, making prototyping easier. True or False?",
        "Single Board Computers like the Raspberry Pi typically run a full operating system like Linux, while microcontrollers like Arduino run only one program uploaded directly to their memory. True or False?",
        "In the microcontroller world, LoRa wireless communication is very popular for long-range IoT projects because its range can reach several kilometers with low power consumption. True or False?"
    ]
}

# Flatten all questions into a single list for easier processing
all_questions = []
for category, questions in test_questions.items():
    for q in questions:
        all_questions.append({
            'category': category,
            'question': q
        })

print(f"\n📝 Test Questions Loaded:")
for category, questions in test_questions.items():
    print(f"  {category}: {len(questions)} questions")
print(f"\nTotal: {len(all_questions)} questions")

# Save questions to file
questions_file = DATA_DIR / 'test_questions.json'
with open(questions_file, 'w') as f:
    json.dump(test_questions, f, indent=2)
print(f"\n✓ Questions saved to: {questions_file}")

---
## 4. Prepare Reference Answers (Ground Truth)

Untuk evaluasi kualitas jawaban, kita perlu reference answers yang akan dijadikan ground truth.

In [ ]:
# Reference answers for evaluation
# These are concise, accurate answers that will be used as ground truth

reference_answers = {
    # Simple questions
    "What is the main function of a resistor in an electronic circuit?": 
        "A resistor limits or regulates the flow of electrical current in a circuit, controlling voltage and current levels according to Ohm's Law.",
    
    "Name three examples of sensors commonly used with Arduino microcontrollers.": 
        "Common sensors include temperature sensors (DHT11/DHT22), ultrasonic distance sensors (HC-SR04), and PIR motion sensors.",
    
    "What does LED stand for, and how do you safely connect it to a voltage source?": 
        "LED stands for Light Emitting Diode. To safely connect it, use a current-limiting resistor in series to prevent excessive current that could damage the LED.",
    
    "What is the basic difference between Voltage and Current?": 
        "Voltage is the electrical potential difference (measured in Volts) that pushes electrons, while current is the actual flow of electrons (measured in Amperes) through a conductor.",
    
    "Which microcontroller is more commonly used by beginners, Arduino Uno or ESP32? Give one reason.": 
        "Arduino Uno is more common for beginners because it has simpler programming, extensive documentation, and a large community support base.",
    
    # Complex questions
    "Conceptually explain how Pulse Width Modulation (PWM) technique can be used to adjust LED brightness or DC motor speed. What is the relationship between duty cycle and the resulting output?": 
        "PWM rapidly switches the output between ON and OFF states. The duty cycle (percentage of time ON) determines the average power delivered. Higher duty cycle means brighter LED or faster motor speed, as the average voltage/power increases proportionally with duty cycle.",
    
    "In the context of serial communication between a microcontroller and a computer, what are the fundamental differences between the UART and I2C protocols in terms of connection topology, number of wires, and speed?": 
        "UART uses point-to-point topology with 2 wires (TX, RX) and typically runs at speeds up to 115200 baud. I2C uses a bus topology with 2 wires (SDA, SCL) allowing multiple devices, supporting speeds from 100kHz to 400kHz (standard/fast mode).",
    
    "A circuit uses a linear regulator LM7805 to step down voltage from 12V to 5V. If the load draws 500mA of current, calculate the approximate power dissipated as heat in the regulator. Why is a switching regulator (like LM2596) more efficient for this case?": 
        "Power dissipated = (Vin - Vout) × I = (12V - 5V) × 0.5A = 3.5W as heat. Switching regulators are more efficient because they use switching techniques instead of linear voltage drop, achieving 80-95% efficiency versus 40-50% for linear regulators.",
    
    "You want to read an analog voltage from a potentiometer using an Arduino. Describe the process flow from the voltage at the analog pin until it becomes a digital value (ADC), including the concept of resolution (e.g., 10-bit).": 
        "The analog voltage (0-5V) enters the ADC, which samples and compares it against a reference. With 10-bit resolution, the voltage range is divided into 1024 levels (2^10). The ADC converts the voltage to a digital value from 0-1023, where each step represents approximately 4.88mV (5V/1024).",
    
    "What is meant by Interrupt in microcontroller programming? Provide a practical example where using an interrupt is more effective than polling when reading input from a button.": 
        "An interrupt is a signal that temporarily halts the current program to execute a specific function (ISR). For button input, interrupts are more effective than polling because they respond immediately to button presses without constantly checking the button state, saving CPU cycles and ensuring no missed events.",
    
    # True/False Basic
    "Capacitors can store electric charge and are often used to smooth out voltage ripples. True or False?": 
        "True. Capacitors store electrical charge and are commonly used in power supply circuits to filter and smooth voltage ripples.",
    
    "An NPN transistor turns ON when the base voltage is lower than the emitter voltage. True or False?": 
        "False. An NPN transistor turns ON when the base voltage is higher than the emitter voltage by approximately 0.7V.",
    
    "Digital pins on a microcontroller can only output HIGH (5V) or LOW (0V). True or False?": 
        "True. Digital pins operate in binary states, outputting either HIGH (typically 5V or 3.3V depending on the microcontroller) or LOW (0V).",
    
    "Ohm's Law states that Current (I) is directly proportional to Voltage (V) and inversely proportional to Resistance (R), formulated as I = V x R. True or False?": 
        "False. The correct formula is I = V / R, where current is voltage divided by resistance.",
    
    "The higher the resistor value connected in series with an LED, the dimmer the LED will glow. True or False?": 
        "True. Higher resistance reduces current flow through the LED, resulting in dimmer light output.",
    
    # True/False Advanced
    "The ESP32 platform from Espressif not only has WiFi and Bluetooth capabilities but also supports programming with MicroPython and Rust, in addition to the Arduino IDE. True or False?": 
        "True. ESP32 supports multiple programming frameworks including Arduino IDE, MicroPython, and Rust (via esp-rs).",
    
    "In modern IoT development, the MQTT protocol has completely replaced HTTP for device-to-cloud communication due to its much higher speed. True or False?": 
        "False. While MQTT is popular for IoT due to its lightweight nature and pub/sub model, HTTP is still widely used. MQTT's advantage is not necessarily higher speed but lower overhead and better efficiency for resource-constrained devices.",
    
    "To connect external sensors or modules to a microcontroller, breakout boards are often used because they already include the necessary support components, making prototyping easier. True or False?": 
        "True. Breakout boards include pull-up resistors, voltage regulators, and other support components, simplifying connections and prototyping.",
    
    "Single Board Computers like the Raspberry Pi typically run a full operating system like Linux, while microcontrollers like Arduino run only one program uploaded directly to their memory. True or False?": 
        "True. Raspberry Pi runs complete operating systems (like Raspberry Pi OS/Linux), while Arduino microcontrollers execute a single program (sketch) from flash memory without an OS.",
    
    "In the microcontroller world, LoRa wireless communication is very popular for long-range IoT projects because its range can reach several kilometers with low power consumption. True or False?": 
        "True. LoRa technology can achieve ranges of 2-15 kilometers in open areas while maintaining low power consumption, making it ideal for long-range IoT applications."
}

print(f"\n📚 Reference Answers Prepared: {len(reference_answers)} answers")

# Save reference answers
ref_answers_file = DATA_DIR / 'reference_answers.json'
with open(ref_answers_file, 'w') as f:
    json.dump(reference_answers, f, indent=2)
print(f"✓ Reference answers saved to: {ref_answers_file}")

---
## 5. Verify System Components

Sebelum memulai testing, kita perlu memverifikasi bahwa semua komponen sistem berjalan dengan baik.

In [ ]:
import requests
import subprocess

def check_ollama_server():
    """Check if Ollama server is running"""
    try:
        response = requests.get("http://127.0.0.1:11434/api/tags", timeout=5)
        if response.status_code == 200:
            models = response.json().get('models', [])
            print("✓ Ollama server is running")
            print(f"  Available models: {len(models)}")
            model_names = [m['name'] for m in models]
            if 'llama3.2:3b' in model_names:
                print("  ✓ llama3.2:3b model found")
                return True
            else:
                print(f"  ⚠ llama3.2:3b not found. Available: {model_names}")
                return False
    except Exception as e:
        print(f"✗ Ollama server not running: {e}")
        print("  Please start with: ollama serve")
        return False

def check_whisper_model():
    """Check if Whisper model is available"""
    try:
        import whisper
        model = whisper.load_model("tiny")
        print("✓ Whisper model loaded successfully")
        return True
    except Exception as e:
        print(f"✗ Whisper model error: {e}")
        return False

def check_embedding_model():
    """Check if embedding model is available"""
    try:
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer('all-MiniLM-L6-v2')
        print("✓ Embedding model loaded successfully")
        return True
    except Exception as e:
        print(f"✗ Embedding model error: {e}")
        return False

def check_piper_tts():
    """Check if Piper TTS is available"""
    piper_bin = Path("/home/rangga/piper/build/piper")
    piper_model = Path("/usr/local/share/piper/models/en_US-lessac-medium.onnx")
    
    if piper_bin.exists() and piper_model.exists():
        print("✓ Piper TTS binary and model found")
        return True
    else:
        print("✗ Piper TTS not properly installed")
        if not piper_bin.exists():
            print(f"  Missing: {piper_bin}")
        if not piper_model.exists():
            print(f"  Missing: {piper_model}")
        return False

print("🔍 Checking System Components...\n")
print("=" * 50)

components_ok = [
    check_ollama_server(),
    check_whisper_model(),
    check_embedding_model(),
    check_piper_tts()
]

print("=" * 50)

if all(components_ok):
    print("\n✅ All components are ready!")
else:
    print("\n⚠️ Some components need attention. Please fix before proceeding.")

---
## 6. Check Reference Documents

Verifikasi bahwa dokumen PDF untuk RAG tersedia.

In [ ]:
# Check for PDF reference documents
pdf_files = ['referensi.pdf', 'jurnal_tambahan.pdf']
pdf_status = {}

print("📄 Checking Reference Documents...\n")

for pdf_file in pdf_files:
    pdf_path = PROJECT_DIR / pdf_file
    if pdf_path.exists():
        file_size = pdf_path.stat().st_size / 1024  # KB
        print(f"✓ {pdf_file} found ({file_size:.1f} KB)")
        pdf_status[pdf_file] = True
    else:
        print(f"✗ {pdf_file} NOT found")
        pdf_status[pdf_file] = False

print(f"\nPDF Status: {sum(pdf_status.values())}/{len(pdf_files)} files available")

---
## 7. Summary and Next Steps

In [ ]:
# Create summary
summary = {
    'setup_date': datetime.now().isoformat(),
    'platform': CONFIG['platform'],
    'total_questions': len(all_questions),
    'question_categories': {cat: len(qs) for cat, qs in test_questions.items()},
    'test_repetitions': CONFIG['num_test_repetitions'],
    'total_test_runs': len(all_questions) * CONFIG['num_test_repetitions'] * 2,  # x2 for RAG and non-RAG
    'reference_answers': len(reference_answers),
    'components_status': all(components_ok),
    'pdf_documents_available': sum(pdf_status.values())
}

print("\n" + "="*60)
print("📊 SETUP SUMMARY")
print("="*60)
for key, value in summary.items():
    print(f"{key}: {value}")

# Save summary
summary_file = RESULTS_DIR / 'setup_summary.json'
with open(summary_file, 'w') as f:
    json.dump(summary, f, indent=2)

print(f"\n✓ Summary saved to: {summary_file}")

print("\n" + "="*60)
print("✅ PREPARATION COMPLETE!")
print("="*60)
print("\nNext Steps:")
print("1. Open notebook: 02_Testing_LLM_RAG.ipynb")
print("2. Run comprehensive testing")
print("3. Analyze results in: 03_Analysis_and_Visualization.ipynb")